# Environment Inspection

Claude operates **blindly** — it can't see the effect of an action unless you show it. An effective agent must **observe the results of its actions** and adapt. This is a design principle more than a new mechanic, but it's the difference between a blind command-executor and an agent that actually works.

**Examples:**
- **Computer use** — after every click/keystroke, Claude gets a **screenshot**. A click could open a menu, navigate, or do nothing; without seeing the new state it can't know.
- **Files** — **read before you write.** To add a route to a file, Claude first *reads* it to learn the structure, then edits safely without breaking things.

> Runnable illustration of read-before-write below (in-memory "file", no disk writes). On Haiku 4.5.

## Guiding inspection via the system prompt

For complex tasks you *instruct* Claude to verify. A video agent's system prompt might say:

- Run `whisper.cpp` (via `bash`) to generate timestamped captions and **verify dialogue placement**.
- Use FFmpeg to **extract screenshots** at intervals and visually inspect the output.
- **Compare** the generated content against the original requirements before finishing.

The pattern: after acting, gather evidence (captions, screenshots, API responses) and check it against the goal.

## Runnable demo — read before write

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-haiku-4-5-20251001"

# An in-memory "file" the agent can read and edit (no disk writes)
FILE = """from flask import Flask

app = Flask(__name__)


@app.route("/")
def index():
    return "Home"
"""


def read_file():
    return FILE

def edit_file(old_str, new_str):
    global FILE
    if old_str not in FILE:
        raise ValueError("old_str not found - read the file first")
    FILE = FILE.replace(old_str, new_str)
    return "edited"

In [2]:
def get_text(message):
    return "\n".join(b.text for b in message.content if b.type == "text")

def add_user_message(messages, m):
    messages.append({"role": "user", "content": m.content if isinstance(m, Message) else m})

def add_assistant_message(messages, m):
    messages.append({"role": "assistant", "content": m.content if isinstance(m, Message) else m})


read_file_schema = ToolParam({
    "name": "read_file",
    "description": "Read and return the full current contents of the file. Call this before editing so you know the exact text to match.",
    "input_schema": {"type": "object", "properties": {}, "required": []},
})

edit_file_schema = ToolParam({
    "name": "edit_file",
    "description": "Replace an exact substring in the file with new text. old_str must match the file exactly (read the file first).",
    "input_schema": {"type": "object", "properties": {
        "old_str": {"type": "string", "description": "Exact text to replace"},
        "new_str": {"type": "string", "description": "Replacement text"}}, "required": ["old_str", "new_str"]},
})

TOOLS = [read_file_schema, edit_file_schema]
DISPATCH = {"read_file": read_file, "edit_file": edit_file}


def run_agent(user_text):
    messages = []
    add_user_message(messages, user_text)
    while True:
        resp = client.messages.create(model=model, max_tokens=1000, temperature=1.0,
                                       messages=messages, tools=TOOLS)
        add_assistant_message(messages, resp)
        for b in resp.content:
            if b.type == "tool_use":
                shown = {k: (v[:40] + "..." if isinstance(v, str) and len(v) > 40 else v) for k, v in b.input.items()}
                print(f"  -> {b.name}({shown})")
        if resp.stop_reason != "tool_use":
            return get_text(resp)
        results = []
        for req in [b for b in resp.content if b.type == "tool_use"]:
            try:
                out = DISPATCH[req.name](**req.input)
                results.append({"type": "tool_result", "tool_use_id": req.id, "content": json.dumps(out), "is_error": False})
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": req.id, "content": f"Error: {e}", "is_error": True})
        add_user_message(messages, results)

In [3]:
print("BEFORE:\n" + FILE)
print("--- agent runs ---")
answer = run_agent("Add a GET route '/health' that returns the string 'ok'. Match the existing style.")
print("--- done ---\n")
print("AFTER:\n" + FILE)

BEFORE:
from flask import Flask

app = Flask(__name__)


@app.route("/")
def index():
    return "Home"

--- agent runs ---
  -> read_file({})
  -> edit_file({'old_str': '@app.route(\\"/\\")\\ndef index():\\n    ret...', 'new_str': '@app.route(\\"/\\")\\ndef index():\\n    ret...'})
  -> read_file({})
  -> edit_file({'old_str': '@app.route(\\"/\\")\\ndef index():\\n    ret...', 'new_str': '@app.route(\\"/\\")\\ndef index():\\n    ret...'})
  -> edit_file({'old_str': '    return "Home"\n', 'new_str': '    return "Home"\n\n\n@app.route("/health"...'})
--- done ---

AFTER:
from flask import Flask

app = Flask(__name__)


@app.route("/")
def index():
    return "Home"


@app.route("/health")
def health():
    return "ok"



Notice the tool order: the agent calls **`read_file` before `edit_file`** — it inspects the current contents (to get an exact `old_str` and match the style) before mutating. That's environment inspection in miniature; without the read, its edit would be a blind guess (and `edit_file` would error if `old_str` didn't match).

## Benefits & the design question

Inspection buys you: **progress tracking**, **error handling** (detect + correct surprises), **quality assurance** (verify before declaring done), and **adaptive behavior**.

When designing any agent, keep asking: **"How will Claude know if this action worked?"** Then give it the tools/instructions to find out —

- read file contents before modifying,
- screenshot after UI interactions,
- check API responses for the expected data,
- validate generated content against requirements.

That's what turns Claude from a blind executor into an agent that understands and adapts to its environment.

Next: **Workflows vs agents** — the trade-offs (reliability, cost, control) that decide which to reach for.